In [1]:
import camelot
import pandas as pd
from PyPDF2 import PdfReader
import re
import numpy as np

In [60]:
pdf_to_read = "Remittance_Cenco.pdf"

# -------------------------
# Step 1: Extraer todas las tablas excepto la última página (1-25)
# -------------------------
tables = camelot.read_pdf(pdf_to_read, pages='1-25', flavor='stream', strip_text='\n')

# -------------------------
# Step 2: Concatenar todas las tablas detectadas
# -------------------------
df_all = pd.concat([table.df for table in tables], ignore_index=True)

# -------------------------
# Step 3: Detectar la fila de encabezado real (la que contiene "VOUCHER")
# -------------------------
header_row_idx = df_all[df_all.apply(lambda row: row.astype(str).str.contains('VOUCHER').any(), axis=1)].index[0]

# Usar esa fila como encabezado
df_all.columns = df_all.iloc[header_row_idx]

# Eliminar la fila usada como encabezado y todas anteriores
df_all = df_all.drop(index=list(range(header_row_idx + 1))).reset_index(drop=True)

# Eliminar cualquier fila idéntica al encabezado
df_all = df_all[~df_all.apply(lambda row: all(row.astype(str) == df_all.columns.astype(str)), axis=1)].reset_index(drop=True)

# -------------------------
# Columnas de referencia
# -------------------------
ref_columns = df_all.columns.tolist()

# -------------------------
# Step 4: Extraer la página 26 con Camelot lattice
# -------------------------
# -------------------------
# Leer página 26 como texto
# -------------------------
reader = PdfReader(pdf_to_read)
last_page_text = reader.pages[25].extract_text()  # página 26, index 25

lines = last_page_text.split('\n')

# Códigos válidos para la primera columna
filter_values = ['CH','DAV','DCA','DCC','DCF','DEV','DND','DPC','FPM','FS','LTG','RPL','VOUCHER']

# Valores posibles de la columna DESCRIPCION
descripcion_values = [
    "FACTURA PROVEEDOR",
    "Costo de Transferen",
    "DESCUENTO APERTURA",
    "DESCUENTO CALENDARI",
    "DESCUENTO COMPRAS",
    "DSTO COMERCIAL FIJO",
    "DEVOLUCION MERCANCI AAAA-",
    "DEVOLUCIONES",
    "DESCUENTO EN PRIMER",
    "FACTURA VENTA",
    "DIF. COSTO/CANTIDAD",
    "DESCUENTO"
]

records = []

for line in lines:
    line_strip = line.strip()
    # Verificar que la línea comience con un código válido
    code_match = next((code for code in filter_values if line_strip.startswith(code)), None)
    if code_match:
        # Buscar DESCRIPCION
        descripcion_match = next((desc for desc in descripcion_values if desc in line_strip), None)
        if descripcion_match:
            voucher = code_match
            descripcion = descripcion_match
            # Tomar el resto de la línea después de DESCRIPCION
            rest = line_strip[line_strip.find(descripcion_match)+len(descripcion_match):].strip()
            # Separar valores numéricos u otros por 2 o más espacios
            numeric_values = re.split(r'\s{2,}', rest)
            # Construir fila alineada con ref_columns
            row_values = []
            for idx, col in enumerate(ref_columns):
                if idx == 0:
                    row_values.append(voucher)
                elif idx == 1:
                    row_values.append(descripcion)
                else:
                    val_idx = idx - 2
                    if val_idx < len(numeric_values):
                        row_values.append(numeric_values[val_idx])
                    else:
                        row_values.append('')
            records.append(row_values)

# Crear df_last con los registros de la página 26
if records:
    df_last = pd.DataFrame(records, columns=ref_columns)
    print(f"✅ Página 26 lista para concatenar, registros extraídos: {len(df_last)}")
    # Concatenar con df_all
    df_all = pd.concat([df_all, df_last], ignore_index=True)
else:
    df_last = pd.DataFrame(columns=ref_columns)
    print("⚠️ No se detectaron registros en la página 26")

# -------------------------
# Step 5: Filtrar registros válidos
# -------------------------
filter_values = ['CH','DAV','DCA','DCC','DCF','DEV','DND','DPC','FPM','FS','LTG','RPL','VOUCHER']
filtered_df = df_all[df_all[df_all.columns[0]].isin(filter_values)].copy()

# -------------------------
# Step 6: Ordenar por VOUCHER, FPM y luego otros alfabéticamente
# -------------------------
def sort_key(value):
    if value == 'VOUCHER':
        return '0'
    elif value == 'FPM':
        return '1'
    else:
        return '2' + str(value)

filtered_df['sort_order'] = filtered_df[filtered_df.columns[0]].apply(sort_key)
filtered_df = filtered_df.sort_values(by='sort_order').drop(columns='sort_order').reset_index(drop=True)

# -------------------------
# Step 7: Eliminar duplicados
# -------------------------
filtered_df = filtered_df.drop_duplicates().reset_index(drop=True)

/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.2204959568734)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.2112062663186)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (301.0, 448.924, 438.37600000000003, 567.215777188329)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/svonbergen/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/camelot/parsers/base.py:238: UserWarning: No tables fou

✅ Página 26 lista para concatenar, registros extraídos: 2


In [88]:
#filtered_df = filtered_df[filtered_df['VOUCHER'] == 'FPM']

In [89]:
filtered_df = filtered_df[["DESCRIPCION", "DOCUMENTO", "VALOR PAG"]]

In [90]:
filtered_df["DESCRIPCION"] = filtered_df["DESCRIPCION"].str.replace(
    "FACTURA PROVEEDOR",
    "Factura",
    case=False,
    regex=False
)

In [91]:
remittance = filtered_df

In [92]:
remittance = remittance.rename(columns={
    "DESCRIPCION": "Tipo de Documento",
    "DOCUMENTO": "Referencia / Factura",
    "VALOR PAG": "Importe de factura"
})

In [93]:
remittance = remittance[remittance["Importe de factura"].astype(str).str.strip() != ""]

In [94]:
remittance["Importe de factura"] = remittance["Importe de factura"].astype(str)


In [95]:
remittance["Importe de factura"] = (
    remittance["Importe de factura"]
      .str.replace(".", "", regex=False)   # Quita los puntos de miles
      .astype(float)                        # Convierte a float
      * -1                                  # Cambia el signo
)

In [96]:
FBL5N = pd.read_excel("FBL5N_Cenco.xlsx", usecols=["Document Type", "Reference", "Amount in local currency"])
FBL5N = FBL5N[FBL5N["Document Type"] == "RV"]
FBL5N = FBL5N.rename(columns={
    "Reference": "Referencia / Factura",
    "Amount in local currency": "importe_FBL5N"
}).reset_index(drop=True)

In [97]:
# --- Paso 3: Merge ---
hrc_template = pd.merge(remittance, FBL5N, on="Referencia / Factura", how="left")
hrc_template["Diferencia"] = pd.NA
hrc_template.loc[
    hrc_template["Tipo de Documento"] == "Factura", 
    "Diferencia"
] = (
    hrc_template["importe_FBL5N"] - hrc_template["Importe de factura"]
)

In [98]:
    diferencias = hrc_template[hrc_template["Diferencia"].notna() & (hrc_template["Diferencia"] != 0)].copy()
    registros_diferencias = pd.DataFrame({
        "Tipo de Documento": "Descuentos no asociados a FC",
        "Referencia / Factura": diferencias["Referencia / Factura"],
        "Importe de factura": diferencias["Diferencia"],
        "Pago Neto": "",
        "Descuento": "MENORES VALORES",
        "Motivo del descuento": np.select(
            condlist=[
                (diferencias["Diferencia"] <= -20000) | (diferencias["Diferencia"] >= 20000),
                (diferencias["Diferencia"].between(-20000, 0, inclusive="neither")),
                (diferencias["Diferencia"].between(0, 20000, inclusive="left"))
            ],
            choicelist=["987", "WOB", "384"],
            default="Error (Revisar)"
        )
    })


In [99]:
hrc_template = pd.concat([hrc_template, registros_diferencias], ignore_index=True)

In [100]:
hrc_template.head()

,Tipo de Documento,Referencia / Factura,Importe de factura,Document Type,importe_FBL5N,Diferencia,Pago Neto,Descuento,Motivo del descuento
0,Factura,PMP1261243,682298.0,RV,682298.00,0.0,NaN,NaN,NaN
1,Factura,PMP1260881,4914495.0,RV,4914494.90,-0.1,NaN,NaN,NaN
2,Factura,PMP1261044,6046853.0,RV,6046852.00,-1.0,NaN,NaN,NaN
3,Factura,PMP1261046,6047100.0,RV,6047101.00,1.0,NaN,NaN,NaN
4,Factura,PMP1260884,11742583.0,RV,11742583.79,0.79,NaN,NaN,NaN


In [101]:
    # --- Paso 4: Comentarios y campos finales ---
    hrc_template["Comentarios"] = np.where(
        hrc_template["Tipo de Documento"] == "Factura", "",
        np.where(
            hrc_template["Descuento"] == "MENORES VALORES",
            hrc_template["Descuento"],
            hrc_template["Descuento"].fillna("") + " " + hrc_template["Referencia / Factura"].fillna("")
        )
    )
    cond_1 = (hrc_template["Motivo del descuento"] == "987") & (hrc_template["Importe de factura"] < -20000)
    cond_2 = (hrc_template["Motivo del descuento"] == "987") & (hrc_template["Importe de factura"] > 20000)
    hrc_template.loc[cond_1, "Comentarios"] = "Myr Vlr Pagado " + hrc_template.loc[cond_1, "Referencia / Factura"].fillna("")
    hrc_template.loc[cond_2, "Comentarios"] = "Saldo FC " + hrc_template.loc[cond_2, "Referencia / Factura"].fillna("")
    
    hrc_template["Pago Neto"] = hrc_template["Importe de factura"]

    columnas_finales = ["Tipo de Documento", "Referencia / Factura", "Importe de factura",
                        "Descuento", "Motivo del descuento", "Pago Neto", "Comentarios"]
    hrc_template = hrc_template[columnas_finales]

In [103]:
hrc_template.to_csv("template_cenco.csv", index=False)